# Grid UCI BNN — Sampler Investigation

Loads **all available samplers** for one `(dataset, split)` and compares them across:
metrics, sparsity, calibration, posterior noise, ESS, and predictive intervals.

Adapted from `uci_investigate.ipynb` for the `gpu_friendly` tree: `grid_boomerang` / `grid_sticky_boomerang` / `nuts` / `nuts_horseshoe` instead of zigzag/boomerang x sticky x PLI, and noise is learned by every sampler here (no fixed-noise variant, so the old notebook's `LEARNED_NOISE` filter and `target_cache` keyed by bool are gone).

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 7. Skeleton dynamics — how the samplers *act*

Everything above works from the resampled `samples` draws (`results/grid/uci_bnn/...`)
and asks "how good is the posterior approximation". This section instead loads the raw
skeleton files from `uci_bnn_grid_skeleton.py` (`results/grid/uci_bnn_skeleton/...`) and
asks "what is the sampler actually doing, mechanically, along its trajectory" — no
predictive metrics, no flashing numbers, just event-level behaviour.

Two data sources per run, at different granularities:
- **`positions`/`velocities`/`times`/`grid_t_max_log`** — one entry per *accepted skeleton
  point* (length `n_skeleton_saved`, the last `N_SKELETON_SAVE` events of the run).
- **`diagnostics`** — one row per *sampler-loop iteration* (bounce, no_event, freeze, thaw,
  or refresh). This is finer-grained: freeze/thaw/no_event iterations don't always advance
  the skeleton index `n` (see `grid_sticky_boomerang.sample`), so `len(diagnostics) !=
  n_skeleton_saved` in general and the two are **not index-aligned** — always analyze them
  independently, never zip them together.

Only present in files produced *after* the `diagnostics`-saving fix — regenerate skeleton
`.pt` files with the current `uci_bnn_grid_skeleton.py` if a run shows `no diagnostics saved`
below.

In [ ]:
DATASET  = "boston"
SPLIT_ID = 0
import pandas as pd
SKELETON_DIR = Path("results/grid/uci_bnn_skeleton")


# Display-name overrides (stem -> label)
LABELS = {
    "grid_zigzag":           "Grid ZigZag",
    "grid_sticky_zigzag":    "Grid Sticky ZigZag",
    "grid_boomerang":        "Grid Boomerang",
    "grid_sticky_boomerang": "Grid Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
}

# Colour palette -- one colour per base sampler family
COLORS = {
    "grid_zigzag":           "#1F77B4",
    "grid_sticky_zigzag":    "#1F77B4",
    "grid_boomerang":        "#FA5C00",
    "grid_sticky_boomerang": "#FA5C00",
    "nuts":                  "#0CCA38",
    "nuts_horseshoe":        "#0CCA38",
}

# Linestyle: solid for sticky/HS, dashed for vanilla
LINESTYLES = {
    "grid_zigzag":           "--",
    "grid_sticky_zigzag":    "-",
    "grid_boomerang":        "--",
    "grid_sticky_boomerang": "-",
    "nuts":                  "--",
    "nuts_horseshoe":        "-",
}

In [ ]:
skel_split_dir = SKELETON_DIR / DATASET / f"split_{SPLIT_ID:02d}"

skel_files = sorted(skel_split_dir.glob("*_skeleton.pt"))
print(skel_split_dir)
assert skel_files, f"No skeleton .pt files found in {skel_split_dir}."

skeletons: dict[str, dict] = {}
for p in skel_files:
    stem = p.stem.removesuffix("_skeleton")
    payload = torch.load(p, map_location="cpu", weights_only=False)
    diag_log = payload.get("diagnostics")
    skeletons[stem] = dict(
        payload = payload,
        diag_df = pd.DataFrame(diag_log) if diag_log else None,
        label   = LABELS.get(stem, stem),
        color   = COLORS.get(stem, "grey"),
        ls      = LINESTYLES.get(stem, "-"),
    )
    n_saved = payload["positions"].shape[0]
    n_diag  = len(diag_log) if diag_log else 0
    tag = "" if diag_log else "  (no diagnostics saved — re-run to backfill)"
    print(f"  [{stem}] {n_saved} skeleton pts, {n_diag} diagnostic rows{tag}")

skeleton_order = list(skeletons.keys())

### 7a. Grid horizon (`t_max`) over the run

`grid_t_max_log[i]` is the adaptive Algorithm-4 horizon used to draw skeleton point `i` —
the window over which the grid Poisson-thinning bound was constructed. Watching it settle
(or oscillate, or drift) tells you how the local curvature of the BNN posterior is behaving
along the trajectory: a horizon that collapses and pins at its floor means the target is
locally very curved (bound has to stay tight everywhere); one that grows and stays large
means the sampler found a relatively flat/well-scaled region.

Two views: against **event index** (so runs with different total event counts still overlay
cleanly) and against **simulation time** (so you see how much physical trajectory length
each regime actually spans — a sampler that racks up many events per unit time is paying a
lot of grid-refinement overhead there).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for stem in skeleton_order:
    sk = skeletons[stem]
    t_max = np.asarray(sk["payload"]["grid_t_max_log"])
    idx   = np.arange(len(t_max))

    axes[0].plot(idx, t_max, color=sk["color"], ls=sk["ls"], lw=1.1,
                 alpha=0.85, label=sk["label"])

    # grid_t_max_log and diagnostics are both logged once per sampler-loop
    # iteration (NOT per skeleton point -- see section 7 note), so
    # diag_df["time"] is the correct x-axis here, not payload["times"]
    # (which is one entry per accepted skeleton point).
    df = sk["diag_df"]
    if df is not None and len(df) == len(t_max):
        axes[1].plot(df["time"].to_numpy(), t_max, color=sk["color"], ls=sk["ls"],
                     lw=1.1, alpha=0.85, label=sk["label"])
    else:
        axes[1].plot(idx, t_max, color=sk["color"], ls=sk["ls"], lw=1.1,
                     alpha=0.4, label=f"{sk['label']} (no diagnostics — vs. index)")

axes[0].set_xlabel("Loop-iteration index (last N_SKELETON_SAVE iterations)")
axes[0].set_ylabel(r"$t_{max}$ (grid horizon)")
axes[0].set_yscale("log")
axes[0].set_title(f"{DATASET.capitalize()} — grid horizon vs. iteration index")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Simulation time")
axes[1].set_ylabel(r"$t_{max}$ (grid horizon)")
axes[1].set_yscale("log")
axes[1].set_title(f"{DATASET.capitalize()} — grid horizon vs. simulation time")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

for stem in skeleton_order:
    t_max = np.asarray(skeletons[stem]["payload"]["grid_t_max_log"])
    print(f"{skeletons[stem]['label']}: t_max mean={t_max.mean():.4g}  "
          f"median={np.median(t_max):.4g}  min={t_max.min():.4g}  max={t_max.max():.4g}  "
          f"final={t_max[-1]:.4g}")

### 7b. Event-type breakdown — bounce / no-event / freeze / thaw / refresh

Every sampler-loop iteration ends in exactly one `event_type`:
- **`bounce`** — a Poisson-thinning candidate was accepted; velocity reflects.
- **`no_event`** — thinning rejected every candidate over the horizon; nothing happened,
  the sampler just advanced its horizon and tried again (pure overhead).
- **`freeze`** (sticky only) — a coordinate hit zero and got stuck there.
- **`thaw`** (sticky only) — a frozen coordinate's exponential clock fired and it reactivated.
- **`refresh`** (Boomerang family only) — the refreshment Poisson clock fired; velocity
  fully resampled.

The `no_event` share is the most direct "wasted work" signal for how well `grid_spacing`/
`grid_t_max_init` are tuned to this BNN target: a high no-event rate means the sampler is
burning gradient evaluations on horizons it keeps rejecting. For the sticky samplers, the
freeze/thaw balance shows whether the chain is actually exploring different sparsity
patterns or has collapsed onto one.

In [ ]:
EVENT_TYPES  = ["bounce", "no_event", "freeze", "thaw", "refresh"]
EVENT_COLORS = {
    "bounce":   "#0CCA38",
    "no_event": "#AAAAAA",
    "freeze":   "#1F77B4",
    "thaw":     "#FA5C00",
    "refresh":  "#9467BD",
}

available = [stem for stem in skeleton_order if skeletons[stem]["diag_df"] is not None]
if not available:
    print("No runs have diagnostics saved yet — nothing to plot here.")
else:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(available))
    bottom = np.zeros(len(available))

    counts_table = []
    for etype in EVENT_TYPES:
        fracs = []
        for stem in available:
            df = skeletons[stem]["diag_df"]
            frac = (df["event_type"] == etype).mean()
            fracs.append(frac)
        fracs = np.array(fracs)
        ax.bar(x, fracs, bottom=bottom, color=EVENT_COLORS[etype], label=etype, width=0.6)
        bottom += fracs
        counts_table.append(fracs)

    ax.set_xticks(x)
    ax.set_xticklabels([skeletons[s]["label"] for s in available], rotation=15, ha="right")
    ax.set_ylabel("Fraction of loop iterations")
    ax.set_title(f"{DATASET.capitalize()} — event-type composition")
    ax.legend(fontsize=8, loc="upper right", bbox_to_anchor=(1.25, 1.0))
    plt.tight_layout()
    plt.show()

    event_frac_df = pd.DataFrame(
        np.array(counts_table).T, index=[skeletons[s]["label"] for s in available], columns=EVENT_TYPES,
    )
    event_frac_df.style.format(precision=4, na_rep="—").background_gradient(cmap="Blues", axis=None)

A static overall ratio hides drift within a run (e.g. the no-event rate settling down as
`t_max` adapts, or freeze/thaw reaching an equilibrium rate only partway through). Rolling
event-type proportions over the iteration window show that evolution directly.

In [ ]:
ROLL_WINDOW = 500

if not available:
    print("No runs have diagnostics saved yet — nothing to plot here.")
else:
    n_types = len(EVENT_TYPES)
    fig, axes = plt.subplots(1, n_types, figsize=(4 * n_types, 3.6), sharex=False)

    for j, etype in enumerate(EVENT_TYPES):
        ax = axes[j]
        for stem in available:
            sk = skeletons[stem]
            df = sk["diag_df"]
            is_type = (df["event_type"] == etype).astype(float)
            if len(is_type) < ROLL_WINDOW:
                continue
            rolling = is_type.rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW // 2).mean()
            ax.plot(rolling.to_numpy(), color=sk["color"], ls=sk["ls"], lw=1.3,
                    alpha=0.9, label=sk["label"])
        ax.set_title(etype)
        ax.set_xlabel("Loop iteration")
        if j == 0:
            ax.set_ylabel(f"Rolling fraction (window={ROLL_WINDOW})")
        ax.set_ylim(-0.02, 1.02)

    axes[-1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f"{DATASET.capitalize()} — event-type rate evolution over the run", y=1.03)
    plt.tight_layout()
    plt.show()

### 7c. Thinning-bound tightness (`max_ratio`)

`max_ratio` is how close the accepted event's true rate came to the upper bound the grid
construction used for Poisson thinning over that horizon (closer to 1 = tight bound, little
wasted proposal probability; near 0 = the bound was very loose relative to the realized
rate, meaning many candidates got thinned away for nothing). This is a *mechanism*
diagnostic, not a correctness one — it tells you how efficiently the sampler's own
acceptance machinery is running, independent of whether the resulting chain mixes well.

In [ ]:
if not available:
    print("No runs have diagnostics saved yet — nothing to plot here.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    for stem in available:
        sk = skeletons[stem]
        df = sk["diag_df"]
        mr = df["max_ratio"].to_numpy()

        axes[0].hist(mr, bins=50, range=(0, 1), density=True, alpha=0.4,
                     color=sk["color"], histtype="stepfilled", label=sk["label"])

        rolling = df["max_ratio"].rolling(ROLL_WINDOW, min_periods=ROLL_WINDOW // 2).mean()
        axes[1].plot(rolling.to_numpy(), color=sk["color"], ls=sk["ls"], lw=1.3,
                     alpha=0.9, label=sk["label"])

    axes[0].set_xlabel("max_ratio (realized rate / bound)")
    axes[0].set_ylabel("Density")
    axes[0].set_title(f"{DATASET.capitalize()} — thinning-bound tightness")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Loop iteration")
    axes[1].set_ylabel(f"Rolling mean max_ratio (window={ROLL_WINDOW})")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f"{DATASET.capitalize()} — bound tightness over the run")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    for stem in available:
        mr = skeletons[stem]["diag_df"]["max_ratio"]
        print(f"{skeletons[stem]['label']}: max_ratio mean={mr.mean():.4f}  median={mr.median():.4f}")

### 7d. Sparsity dynamics (sticky samplers) — freeze/thaw timing

Only meaningful for `grid_sticky_zigzag`/`grid_sticky_boomerang` (`diag_df["sparsity"]` is
`n_frozen / D` at that iteration; plain zigzag/boomerang have no freezing so this stays at
0). Two things worth seeing: the **sparsity trajectory itself** (does it climb to an
equilibrium and stay there, or keep drifting — a sign the chain hasn't settled into a stable
active-coordinate set yet), and **event timing** — a scatter of *when* freeze vs. thaw
events happen shows whether the two are balanced throughout or whether one dominates in a
particular phase (e.g. a "cold start" burst of freezing early on, followed by a slower
thaw-dominated equilibration).

In [ ]:
sticky_available = [stem for stem in available if "sticky" in stem]

if not sticky_available:
    print("No sticky-sampler runs with diagnostics found — nothing to plot here.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    for stem in sticky_available:
        sk = skeletons[stem]
        df = sk["diag_df"]

        axes[0].plot(df["sparsity"].to_numpy(), color=sk["color"], ls=sk["ls"],
                     lw=1.1, alpha=0.85, label=sk["label"])

        freeze_idx = df.index[df["event_type"] == "freeze"].to_numpy()
        thaw_idx   = df.index[df["event_type"] == "thaw"].to_numpy()
        y_freeze = np.full(len(freeze_idx), 1.0)
        y_thaw   = np.full(len(thaw_idx), 0.0)
        axes[1].scatter(freeze_idx, y_freeze, marker="|", color=sk["color"], alpha=0.5, s=200,
                        label=f"{sk['label']} freeze")
        axes[1].scatter(thaw_idx, y_thaw, marker="|", color=sk["color"], alpha=0.5, s=200,
                        label=f"{sk['label']} thaw")

    axes[0].set_xlabel("Loop iteration")
    axes[0].set_ylabel("Sparsity (n_frozen / D)")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"{DATASET.capitalize()} — sparsity trajectory")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Loop iteration")
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(["thaw", "freeze"])
    axes[1].set_title(f"{DATASET.capitalize()} — freeze/thaw event timing")
    axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))

    plt.tight_layout()
    plt.show()

    for stem in sticky_available:
        df = skeletons[stem]["diag_df"]
        n_freeze = (df["event_type"] == "freeze").sum()
        n_thaw   = (df["event_type"] == "thaw").sum()
        print(f"{skeletons[stem]['label']}: {n_freeze} freezes, {n_thaw} thaws, "
              f"final sparsity={df['sparsity'].iloc[-1]:.3f}")

### 7e. ZigZag vs. Boomerang family — mechanism differences

Once `grid_boomerang`/`grid_sticky_boomerang` skeletons are available alongside the zigzag
ones here, the plots above (7a–7d) already overlay every loaded sampler automatically — no
changes needed, just re-run once the new `.pt` files exist in this split's skeleton
directory. A few comparisons that are specifically informative *across* the two families
(not just within one):

- **Bounce rate and no-event rate side by side (7b/7c)** — ZigZag flips one coordinate's
  sign per bounce (a coordinate-wise process); Boomerang reflects the full velocity vector
  and additionally has `refresh` events on its own Poisson clock. Comparing `no_event`
  share tells you which family's grid bound is better matched to this target's curvature;
  comparing `bounce` share alone is less informative since the two processes bounce for
  different reasons.
- **`refresh` share and its rolling rate (7b)** — Boomerang-only. A `refresh_rate` that's
  too low relative to how fast the chain naturally decorrelates via bounces shows up as long
  stretches with no refresh events; too high shows up as refreshes dominating over
  actual boundary-driven bounces (bounces become a rare correction rather than the primary
  exploration mechanism).
- **`t_max` scale and stability (7a)** — Boomerang's grid horizon is anchored to
  `GRID_T_MAX_INIT_BOOM = pi/4` (periodic structure from the harmonic reference dynamics)
  vs. ZigZag's data-driven `GRID_T_MAX_INIT_ZIGZAG` (see `uci_bnn_grid.py`'s docstring on
  why these differ by ~2 orders of magnitude). Comparing the *shape* of convergence (how
  many iterations before `t_max` stabilizes) rather than absolute values is the fair
  cross-family read.
- **Sticky freeze/thaw balance (7d), sticky-ZigZag vs. sticky-Boomerang** — both use the
  same `kappa`/`can_freeze` construction from `cfg.prior_inclusion_weight`, so a difference
  in final sparsity or freeze/thaw event rates between the two sticky variants isolates the
  effect of the underlying velocity process (coordinate-wise vs. harmonic) on how the
  spike-and-slab sparsity pattern is explored, holding the prior fixed.

The cell below is a ready-made summary table across every loaded sampler (event fractions +
mean `t_max` + mean `max_ratio` + final sparsity in one place) — fill in once more samplers
land.

In [ ]:
summary_rows = []
for stem in skeleton_order:
    sk = skeletons[stem]
    df = sk["diag_df"]
    t_max = np.asarray(sk["payload"]["grid_t_max_log"])

    row = {
        "Sampler":  sk["label"],
        "t_max mean": t_max.mean(),
        "t_max final": t_max[-1],
    }
    if df is not None:
        for etype in EVENT_TYPES:
            row[f"% {etype}"] = (df["event_type"] == etype).mean()
        row["mean max_ratio"] = df["max_ratio"].mean()
        row["final sparsity"] = df["sparsity"].iloc[-1] if "sticky" in stem else float("nan")
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Sampler")
summary_df.style.format(precision=4, na_rep="—")

### 7f. The rate itself, $\lambda(t)$, between two skeleton points

Everything above (7a-7e) is a proxy for sampler behaviour derived from *outcomes*
(accepted/rejected event counts, the adapted horizon). This section instead reproduces the
**true rate function itself**: between two consecutive skeleton points $(x_n, v_n) \to
(x_{n+1}, v_{n+1})$, position and velocity evolve along a fixed deterministic flow
(ZigZag: linear; Boomerang: harmonic orbit around $x_{\text{ref}}$), so the Poisson rate
$\lambda(t)$ the grid bound thins against is a genuinely deterministic, closed-form function
of $t \in [0, \Delta t_n]$ anchored at $(x_n, v_n)$ — exactly the sampler's own
`_rate_scalar(t, x, v)` closure, called here directly (off-graph, no autodiff, no
resampling), not re-derived or approximated.

The two families' `_rate_scalar` differ in what they return: **ZigZag's is already the
final non-negative rate** $\sum_j \mathrm{clamp}(v_j \nabla_j U(x_t), 0) + D\gamma$ (always
$\geq D\gamma > 0$, no separate signed version exists). **Boomerang's is the raw signed
inner product** $\langle v_t, \nabla U_{\text{excess}}(x_t)\rangle$, which does cross zero —
`grid_thinning` clamps it at accept-time (`lam_true = max(rate_scalar_fn(t), 0.0)`), so for
Boomerang both the signed value and its clamped/true-rate counterpart are plotted, since the
sign changes explain the true rate's shape.

Non-sticky only (`grid_zigzag`, `grid_boomerang`) for now: the sticky rate additionally
depends on `frozen_mask` at that point in the run, which isn't saved per-skeleton-point
(only `frozen_mask_final`, the end-of-run mask, is) — reproducing it exactly would need
either replaying `diagnostics`' freeze/thaw events from $t=0$, or extending the skeleton
script to save `frozen_mask` per point. Deferred until needed.

Requires rebuilding the `BayesianModule` target for this split (same construction as
sections 1-6) and one grid sampler instance per family, purely to call its `_rate_scalar` —
no sampling is run.

In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import build_zigzag_sampler, build_boomerang_sampler, BNNConfig, make_split, load_raw_datasets, build_target, BASE_SEED, DTYPE, DEVICE

RATE_STEM = "grid_zigzag" if "grid_zigzag" in skeletons else next(
    (s for s in skeleton_order if s in ("grid_zigzag", "grid_boomerang")), None,
)
print(RATE_STEM)
if RATE_STEM is None:
    print("No non-sticky skeleton (grid_zigzag / grid_boomerang) loaded -- nothing to reproduce here.")
else:
    sk = skeletons[RATE_STEM]
    payload = sk["payload"]
    rate_cfg = BNNConfig(
        layer_sizes=payload["layer_sizes"],
        activation=payload["activation"],
        prior_sigma_scale=payload["prior_sigma_scale"],
    )
    rate_data = make_split(*load_raw_datasets((DATASET,))[DATASET],
                            seed=BASE_SEED + payload["split_id"], dtype=DTYPE, device=DEVICE)
    rate_bm, rate_x_ref, rate_Sigma_inv = build_target(rate_data, rate_cfg)

    if RATE_STEM == "grid_boomerang":
        rate_sampler = build_boomerang_sampler(rate_bm, rate_x_ref, rate_Sigma_inv)
    else:
        rate_sampler = build_zigzag_sampler(rate_bm)

    print(f"Rebuilt {skeletons[RATE_STEM]['label']} sampler for rate reproduction (D={rate_bm.D}).")

Pick a handful of representative intervals along the saved skeleton tail (evenly spaced
through it, plus the very first) and evaluate $\lambda(t)$ densely across each one. Each
panel is one interval $[0, \Delta t_n]$, anchored at that interval's own $(x_n, v_n)$ — so
every curve below is the literal function the sampler's grid bound was thinning against at
that point in the run, not a summary statistic of it.

In [ ]:
N_INTERVALS = 6
N_T_POINTS  = 200
from sazz.gpu_friendly.scripts.uci_bnn_grid import build_zigzag_sampler, build_boomerang_sampler


if RATE_STEM is None:
    print("Skipped -- see previous cell.")
else:
    positions = payload["positions"]
    velocities = payload["velocities"]
    times = payload["times"]
    n_pts = positions.shape[0]

    is_boomerang = (RATE_STEM == "grid_boomerang")

    # Evenly spaced interval START indices along the saved tail (skip the
    # very last point -- it has no "next" point to form an interval with).
    interval_idx = np.linspace(0, n_pts - 2, N_INTERVALS).round().astype(int)
    interval_idx = sorted(set(interval_idx.tolist()))

    fig, axes = plt.subplots(1, len(interval_idx), figsize=(4 * len(interval_idx), 3.2), sharey=False)
    if len(interval_idx) == 1:
        axes = [axes]

    for ax, n in zip(axes, interval_idx):
        x_n = positions[n]
        v_n = velocities[n]
        dt_n = float(times[n + 1] - times[n])
        if dt_n <= 0:
            ax.set_title(f"n={n} (dt<=0, skipped)")
            continue

        t_grid = np.linspace(0.0, dt_n, N_T_POINTS)
        raw_rate = np.array([rate_sampler._rate_scalar(float(t), x_n, v_n) for t in t_grid])

        if is_boomerang:
            true_rate = np.clip(raw_rate, 0.0, None)
            ax.plot(t_grid, raw_rate, color="grey", lw=1.0, ls=":", label=r"signed $\lambda(t)$")
            ax.plot(t_grid, true_rate, color=sk["color"], lw=1.6, label=r"$\max(\lambda(t), 0)$")
            ax.axhline(0.0, color="black", lw=0.6, alpha=0.5)
        else:
            ax.plot(t_grid, raw_rate, color=sk["color"], lw=1.6, label=r"$\lambda(t)$ (already $\geq D\gamma$)")

        ax.set_title(f"skeleton interval n={n}\n" + r"$\Delta t=$" + f"{dt_n:.2e}")
        ax.set_xlabel("$t$")
        if n == interval_idx[0]:
            ax.set_ylabel(r"$\lambda(t)$")

    axes[-1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f"{DATASET.capitalize()} — {sk['label']}: true rate between skeleton points", y=1.05)
    plt.tight_layout()
    plt.show()